<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# DX Analytics

Dr Yves J Hilpisch

The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:training@tpq.io) | [@dyjh](http://twitter.com/dyjh)

## Imports

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')

!git clone https://github.com/yhilpisch/dx.git
sys.path.append('dx')


In [ ]:
import dx
import math
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

## Monte Carlo Simulation

In [ ]:
S0 = 36.
K = 40.
T = 1.0
r = 0.06
sigma = 0.2
I = 100000
M = 50

In [ ]:
dt = T / M
dt

In [ ]:
%%time
S = np.zeros((M + 1, I))
S[0] = S0
rand = np.random.standard_normal((M + 1, I))
rand = (rand - rand.mean()) / rand.std()  # moment matching
for t in range(1, M + 1):
    S[t] = S[t - 1] * np.exp((r - sigma ** 2 / 2) * dt +
                             sigma * math.sqrt(dt) * rand[t])

In [ ]:
rand.mean()

In [ ]:
rand.std()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(S[:, :100]);

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(S[-1], bins=35)
plt.axvline(S[-1].mean(), c='r');

In [ ]:
S[-1].mean()

In [ ]:
S0 * math.exp(r * T)

## Monte Carlo Valuation

### European Option

In [ ]:
h = np.maximum(K - S[-1], 0)
h.shape

In [ ]:
V0e = math.exp(-r * T) * h.mean()
V0e

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(h, bins=35, label='payoff')
plt.axvline(V0e, c='r', label='present value')
plt.legend();

### American Option

In [ ]:
h = np.maximum(K - S, 0)
h.shape

In [ ]:
df = math.exp(-r * dt)

In [ ]:
%%time
V = h[-1].copy()
for t in range(M - 1, 0, -1):
    reg = np.polyfit(S[t], V * df, deg=7)
    C = np.polyval(reg, S[t])
    V = np.where(h[t] > C, h[t], V * df)

In [ ]:
V0a = df * V.mean()
V0a

## DX Analytics

In [ ]:
import datetime as dt

In [ ]:
csr = dx.constant_short_rate('r', r)

In [ ]:
c = {'initial_value': S0,
     'strike': K,
     'volatility': sigma,
     'maturity': dt.datetime(2019, 12, 31),
     'final_date': dt.datetime(2019, 12, 31),
     'paths': I,
     'frequency': 'W',
     'currency': 'USD'}

In [ ]:
me = dx.market_environment('me', dt.datetime(2019, 1, 1))

In [ ]:
me.add_curve('discount_curve', csr)

In [ ]:
me.constants.update(c)

In [ ]:
me.constants

In [ ]:
gbm = dx.geometric_brownian_motion('gbm', me)

In [ ]:
%time gbm.get_instrument_values()

In [ ]:
paths = pd.DataFrame(gbm.get_instrument_values(), index=gbm.time_grid)

In [ ]:
paths.iloc[:, :100].plot(figsize=(10, 6), legend=False);

In [ ]:
eur_put = dx.valuation_mcs_european_single('eur_put', gbm, me,
                                'np.maximum(strike - maturity_value, 0)')

In [ ]:
eur_put.present_value()

In [ ]:
eur_put.delta()

In [ ]:
am_put = dx.valuation_mcs_american_single('am_put', gbm, me,
                                'np.maximum(strike - instrument_values, 0)')

In [ ]:
am_put.present_value()

In [ ]:
am_put.delta()

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">